# LangChain Runnables Tutorial

This notebook introduces the core idea behind LangChain runnables: small pieces of logic that can be chained, composed, and reused like building blocks.

A runnable is just a unit that takes an input, processes it, and returns an output. That makes it easy to build workflows for prompts, transformations, parsers, and model calls.

In this lesson, we will cover:
- RunnableLambda for custom functions
- RunnableSequence for ordered pipelines
- RunnableParallel for concurrent work
- RunnableBranch for conditional logic
- composing runnable steps with the pipe operator `|`

> Why this matters: in real AI apps, you often need to clean input, format a prompt, call a model, and parse output. Runnables make that flow readable and reusable.

## 1. What is a Runnable?

A runnable is an object that supports an `invoke()` method. This is the common interface used across many LangChain components.

The main idea:
- An input enters the runnable.
- The runnable performs a transformation.
- The result is returned.

This pattern is especially useful for chaining operations without writing a lot of manual glue code.

Think of a runnable like a function that can be combined with other functions.

In [2]:
from langchain_core.runnables import RunnableLambda, RunnableSequence, RunnableParallel, RunnableBranch
from langchain_core.prompts import PromptTemplate

# 1) Simple runnable using a custom lambda
clean_name = RunnableLambda(lambda x: x.strip().title())
print(clean_name.invoke("  alice   "))

# 2) A runnable that adds a greeting
make_greeting = RunnableLambda(lambda x: f"Hello, {x}!")
print(make_greeting.invoke("Alice"))

c:\Users\sidha\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Alice
Hello, Alice!


## 2. RunnableSequence: build a pipeline

`RunnableSequence` runs steps in order. This is a great pattern for:
- normalizing text
- formatting prompts
- passing a result to another component

The flow is easy to follow because each stage is explicit.

In [12]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableSequence, RunnableParallel

load_dotenv()

prompt1 = PromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

model = ChatOpenAI()

parser = StrOutputParser()

prompt2 = PromptTemplate(
    template='Explain the following joke - {text}',
    input_variables=['text']
)

chain = RunnableSequence(prompt1, model, parser, prompt2, model, parser)

print(chain.invoke({'topic':'AI'}))

This joke plays on the idea of AI (artificial intelligence) and robots having a relationship. In this scenario, the AI (which is typically portrayed as a more advanced and emotionally intelligent entity) breaks up with the robot because it couldn't handle its emotional baggage. This suggests that the robot was struggling with emotions or issues that the AI found overwhelming or difficult to deal with, leading to the breakup. The joke relies on the humor of imagining AI and robots having personal relationships and experiencing emotional problems.


## 3. RunnableParallel: run independent tasks together

`RunnableParallel` is helpful when multiple transformations depend on the same input but can be computed independently.

This pattern is useful when you want:
- uppercase and lowercase versions of text
- multiple summaries or extracted features
- several branches that work on the same data simultaneously

In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableSequence, RunnableParallel

load_dotenv()

prompt1 = PromptTemplate(
    template='Generate a tweet about {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Generate a Linkedin post about {topic}',
    input_variables=['topic']
)

model = ChatOpenAI()

parser = StrOutputParser()

parallel_chain = RunnableParallel({
    'tweet': RunnableSequence(prompt1, model, parser),
    'linkedin': RunnableSequence(prompt2, model, parser)
})

result = parallel_chain.invoke({'topic':'AI'})
print(result['tweet'])
print(result['linkedin'])

Exciting advancements in AI are transforming industries and creating new possibilities for innovation. The power of artificial intelligence is truly limitless! 🤖 #AI #technology #innovation
Exciting news in the world of Artificial Intelligence! AI technology is rapidly advancing and revolutionizing industries across the board. From predictive analytics to autonomous vehicles, the possibilities are endless. How are you incorporating AI into your business strategy? Let's chat about the future of AI together! #ArtificialIntelligence #AI #TechRevolution


### RunnablePassthrough

In [15]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# RunnablePassthrough simply passes the input through unchanged.
# It is useful when you want to preserve the original value while
# also adding transformed or derived outputs in the same pipeline.

passthrough_chain = (
    RunnablePassthrough()
    | RunnableLambda(lambda x: {"original": x, "upper": x.upper(), "lower": x.lower()})
)

result = passthrough_chain.invoke("LangChain Runnables are powerful")
print(result)

{'original': 'LangChain Runnables are powerful', 'upper': 'LANGCHAIN RUNNABLES ARE POWERFUL', 'lower': 'langchain runnables are powerful'}


### RunnableLambda

RunnableLambda is a LangChain wrapper that lets you turn any normal Python function into a runnable step inside a pipeline.

In simple terms:

- a runnable is something that accepts an input and returns an output
- RunnableLambda makes your custom function behave like that
- then you can chain it with prompts, models, parsers, and other runnables

In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableSequence, RunnableLambda, RunnablePassthrough, RunnableParallel

load_dotenv()

def word_count(text):
    return len(text.split())

prompt = PromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

model = ChatOpenAI()

parser = StrOutputParser()

joke_gen_chain = RunnableSequence(prompt, model, parser)

parallel_chain = RunnableParallel({
    'joke': RunnablePassthrough(),
    'word_count': RunnableLambda(word_count)
})

final_chain = RunnableSequence(joke_gen_chain, parallel_chain)

result = final_chain.invoke({'topic':'AI'})

final_result = """{} \n word count - {}""".format(result['joke'], result['word_count'])

print(final_result)

Why did the AI break up with its girlfriend? It couldn't handle the emotional baggage! 
 word count - 15


### 4. RunnableBranch: choose the next action dynamically

Sometimes you want different logic depending on the input. `RunnableBranch` lets you define a set of conditions and pick the correct runnable.

This is useful when:
- a user asks for a summary vs. a detailed answer
- you need different behavior for short vs. long text
- a workflow has several valid execution paths

In real projects, branch logic helps you keep prompts and logic clean without complex if/else nesting.

In [7]:
# Choose behavior based on the text length
branch = RunnableBranch(
    (lambda x: len(x) > 20, RunnableLambda(lambda x: "Detailed answer mode")),
    (lambda x: len(x) <= 20, RunnableLambda(lambda x: "Short answer mode")),
    RunnableLambda(lambda x: "Default fallback mode")
)

print(branch.invoke("This intence."))
print(branch.invoke("This is a much longer sentence that clearly exceeds the short threshold."))

Short answer mode
Detailed answer mode


## 5. Why runnables are so useful

Runnables help you think in terms of composable transformations instead of one giant function.

A common pattern in LangChain apps is:
1. clean input
2. build a prompt
3. run a model
4. parse the result
5. return structured output

Each step can be its own runnable, and the whole workflow becomes easier to test, reuse, and debug.

### Best practices
- Keep each runnable focused on one job.
- Prefer clear names and simple transformations.
- Use `|` to make pipeline flow readable.
- Keep branching logic simple and explicit.
- If your pipeline gets long, split it into smaller runnable steps.

This approach makes your AI workflow more maintainable and easier to extend as requirements change.